In [2]:
import itertools
import pandas as pd
import re

def evaluate_formula(formula, symbol_mapping):


    safe_dict = symbol_mapping.copy()


    processed = formula
    processed = processed.replace("<=>", "==")
    processed = processed.replace("=>", "<=")
    processed = processed.replace("->", "<=")


    processed = re.sub(r'\bimplies\b', '<=', processed)

    try:
        return bool(eval(processed, {"__builtins__": None}, safe_dict))
    except Exception as e:
        raise ValueError(f"Error evaluating formula '{formula}': {e}")

def check_entailment(symbols, kb_rules, query):


    combinations = list(itertools.product([True, False], repeat=len(symbols)))

    rows = []
    entails = True

    for combo in combinations:
        model = dict(zip(symbols, combo))


        kb_evals = []
        for rule in kb_rules:
            kb_evals.append(evaluate_formula(rule, model))


        kb_true = all(kb_evals)


        query_true = evaluate_formula(query, model)

        row_entails = True
        if kb_true and not query_true:
            entails = False
            row_entails = False


        row_data = model.copy()
        for i, rule in enumerate(kb_rules):
            row_data[f"Rule {i+1}"] = kb_evals[i]
        row_data["KB Overall"] = kb_true
        row_data["Query"] = query_true
        row_data["KB => Query"] = row_entails
        rows.append(row_data)

    df = pd.DataFrame(rows)
    return entails, df


import sys

print("--- Propositional Logic Entailment Solver ---")
symbols_input = input("Enter your propositional symbols separated by commas (e.g., P, Q): ")

print("\nEnter your Knowledge Base rules one by one. Press Enter on an empty line when finished.")
kb_input = []
while True:
    rule = input(f"Enter KB Rule {len(kb_input) + 1} (or press Enter to finish): ").strip()
    if not rule:
        break
    kb_input.append(rule)

query_input = input("\nEnter the query you want to test (e.g., Q): ").strip()

symbols = [s.strip() for s in symbols_input.split(",") if s.strip()]

if not symbols or not kb_input or not query_input:
    print("\nError: Symbols, Knowledge Base rules, and Query cannot be empty!")
else:
    print("\n" + "="*40)
    print("Evaluating Entailment...")
    print(f"Symbols: {symbols}")
    print(f"Knowledge Base: {kb_input}")
    print(f"Query: {query_input}")
    print("="*40 + "\n")

    try:
        entails, truth_table = check_entailment(symbols, kb_input, query_input)


        dark_styled_table = truth_table.style.set_properties(**{
            'background-color': '#1e1e1e',
            'color': '#ffffff',
            'border-color': '#444444'
        })
        display(dark_styled_table)

        print("\n" + "="*40)
        if entails:
            print("RESULT: YES, the Knowledge Base entails the query (KB ⊨ Query).")
        else:
            print("RESULT: NO, the Knowledge Base does NOT entail the query (KB ⊭ Query).")
        print("="*40)
    except Exception as e:
        print(f"Error during evaluation: {e}")







--- Propositional Logic Entailment Solver ---
Enter your propositional symbols separated by commas (e.g., P, Q): P,Q

Enter your Knowledge Base rules one by one. Press Enter on an empty line when finished.
Enter KB Rule 1 (or press Enter to finish): P->Q
Enter KB Rule 2 (or press Enter to finish): 

Enter the query you want to test (e.g., Q): Q

Evaluating Entailment...
Symbols: ['P', 'Q']
Knowledge Base: ['P->Q']
Query: Q



,P,Q,Rule 1,KB Overall,Query,KB => Query
0,True,True,True,True,True,True
1,True,False,False,False,False,True
2,False,True,True,True,True,True
3,False,False,True,True,False,False



RESULT: NO, the Knowledge Base does NOT entail the query (KB ⊭ Query).
